In [2]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# --- Column mapping: CSV column -> model feature name ---
COLUMN_MAP = {
    "Flow Duration":      "Flow Duration",
    "Tot Fwd Pkts":       "Total Fwd Packets",
    "Tot Bwd Pkts":       "Total Backward Packets",
    "TotLen Fwd Pkts":    "Total Length of Fwd Packets",
    "Fwd Pkt Len Max":    "Fwd Packet Length Max",
    "Fwd Pkt Len Min":    "Fwd Packet Length Min",
    "Fwd Pkt Len Mean":   "Fwd Packet Length Mean",
    "Fwd Pkt Len Std":    "Fwd Packet Length Std",
    "Bwd Pkt Len Max":    "Bwd Packet Length Max",
    "Bwd Pkt Len Mean":   "Bwd Packet Length Mean",
    "Bwd Pkt Len Std":    "Bwd Packet Length Std",
    "Flow Byts/s":        "Flow Bytes/s",
    "Flow Pkts/s":        "Flow Packets/s",
    "Flow IAT Mean":      "Flow IAT Mean",
    "Flow IAT Std":       "Flow IAT Std",
    "Flow IAT Max":       "Flow IAT Max",
    "Flow IAT Min":       "Flow IAT Min",
    "Fwd IAT Std":        "Fwd IAT Std",
    "Fwd IAT Min":        "Fwd IAT Min",
    "Bwd IAT Tot":        "Bwd IAT Total",
    "Bwd IAT Max":        "Bwd IAT Max",
    "Bwd IAT Min":        "Bwd IAT Min",
    "Fwd Header Len":     "Fwd Header Length",
    "Bwd Header Len":     "Bwd Header Length",
    "Bwd Pkts/s":         "Bwd Packets/s",
    "Pkt Len Min":        "Min Packet Length",
    "Pkt Len Max":        "Max Packet Length",
    "Pkt Len Mean":       "Packet Length Mean",
    "Pkt Len Std":        "Packet Length Std",
    "FIN Flag Cnt":       "FIN Flag Count",
    "PSH Flag Cnt":       "PSH Flag Count",
    "ACK Flag Cnt":       "ACK Flag Count",
    "Init Fwd Win Byts":  "Init_Win_bytes_forward",
    "Init Bwd Win Byts":  "Init_Win_bytes_backward",
    "Fwd Act Data Pkts":  "act_data_pkt_fwd",
    "Fwd Seg Size Min":   "min_seg_size_forward",
    "Active Mean":        "Active Mean",
    "Active Max":         "Active Max",
    "Active Min":         "Active Min",
    "Idle Mean":          "Idle Mean",
}

MODEL_FEATURE_COLUMNS = list(COLUMN_MAP.values())  # preserves the order above

# --- Load artifacts ---
model = joblib.load("../newOutput/ensemble_model.pkl")
scaler = joblib.load("../newOutput/scaler.pkl")
label_encoder = joblib.load("../newOutput/label_encoder.pkl")

# --- Load CSV ---
# df = pd.read_csv("../dataset/IDS2018/02-15-2018/02-15-2018.csv")
df = pd.read_csv('../dataset/IDS2018/FTP_BruteForce/ftpBruteForce.csv')

# Rename only the columns we need
df = df.rename(columns=COLUMN_MAP)

# Select and order the 40 features
X = df[MODEL_FEATURE_COLUMNS].copy()

# Clean: coerce to numeric, replace inf/-inf, fill NaN with 0
X = X.apply(pd.to_numeric, errors="coerce")
X.replace([np.inf, -np.inf], 0, inplace=True)
X.fillna(0, inplace=True)

# Scale
X_scaled = scaler.transform(X.values)

# Predict
y_pred_enc = model.predict(X_scaled)
y_pred = label_encoder.inverse_transform(y_pred_enc)

# Ground truth - adjust the label column name if it differs
y_true = df["Label"].values


In [7]:
LABEL_MAP_3 = {
    "FTP-BruteForce": "Brute Force",
    "SSH-Bruteforce": "Brute Force",
    "Benign": "Normal Traffic",
}

y3_true_mapped = pd.Series(df3["Label"]).map(LABEL_MAP_3).values
print(pd.Series(y3_true_mapped).isna().sum())  # should be 0

df3 = pd.read_csv('../dataset/IDS2018/FTP_BruteForce/ftpBruteForce.csv')

df3 = df3.rename(columns=COLUMN_MAP)

X3 = df3[MODEL_FEATURE_COLUMNS].copy()

X3 = X3.apply(pd.to_numeric, errors="coerce")
X3.replace([np.inf, -np.inf], 0, inplace=True)
X3.fillna(0, inplace=True)

X3_scaled = scaler.transform(X3.values)

y3_pred_enc = model.predict(X3_scaled)
y3_pred = label_encoder.inverse_transform(y3_pred_enc)

y3_true_mapped = pd.Series(df3["Label"]).map(LABEL_MAP_3).values

print(classification_report(y3_true_mapped, y3_pred))

present_labels3 = sorted(set(y3_true_mapped))
cm3 = pd.DataFrame(
    confusion_matrix(y3_true_mapped, y3_pred, labels=present_labels3),
    index=present_labels3,
    columns=present_labels3
)
print(cm3)

results3 = df3[["Label"]].copy()
results3["Mapped Label"] = y3_true_mapped
results3["Predicted"] = y3_pred
results3["Correct"] = results3["Mapped Label"] == results3["Predicted"]
print(f"\nOverall Accuracy: {results3['Correct'].mean():.4f}")

0


c:\Users\abhinav\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\abhinav\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\abhinav\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

                precision    recall  f1-score   support

   Brute Force       0.00      0.00      0.00    380949
          DDoS       0.00      0.00      0.00         0
           DoS       0.00      0.00      0.00         0
Normal Traffic       0.77      1.00      0.87    667626
     Port Scan       0.00      0.00      0.00         0

      accuracy                           0.64   1048575
     macro avg       0.15      0.20      0.17   1048575
  weighted avg       0.49      0.64      0.55   1048575

                Brute Force  Normal Traffic
Brute Force               0          204645
Normal Traffic            2          667147

Overall Accuracy: 0.6362


In [6]:
print(df3["Label"].unique())
print(pd.Series(y3_true_mapped).isna().sum())

<StringArray>
['Benign', 'FTP-BruteForce', 'SSH-Bruteforce']
Length: 3, dtype: str
855215
